In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score
from functions.GINI import gini_xgb
from functions.PSI import calculate_psi
import xgboost as xgb
import optuna

# Model tuning

In [34]:
df_train = pd.read_csv('df_train_op_1.csv',sep=',').reset_index(drop=True)
df_test = pd.read_csv('df_test_op_1.csv',sep=',').reset_index(drop=True)
df_val = pd.read_csv('df_val_op_1.csv',sep=',').reset_index(drop=True)

In [35]:
df_train.columns.tolist()

['MAKH',
 'DATEIN_DATE',
 'MONTHIN',
 'TAG_GB',
 'CNT_COLLATERAL_HCM_QSDD',
 'AVG_LTV_L3M',
 'ca_over_1m_l1m_ct_l3m',
 'ca_min_l12m',
 'sa_min_l12m',
 'sa_median_l12m',
 'casa_min_l12m',
 'sa_min_l6m',
 'casa_min_l6m_ct_l12m',
 'casa_q25_l6m_ct_l12m',
 'casa_median_l6m_ct_l12m',
 'ca_q75_l6m_ct_l12m',
 'EMPLOYMENT_STATUS',
 'dpd_iqr_l3m',
 'dpd_min_l12m',
 'dpd_max_l12m',
 'n_dpd_1_30_l3m',
 'cons_delq_0_l3m',
 'cons_delq_90_l3m',
 'cons_delq_increase_l6m',
 'cons_delq_90_l12m',
 'days_last_ptp_l1m',
 'DIA_CHI']

In [36]:
len(df_train.columns)-4

23

In [37]:
x_train = df_train.loc[:,[col for col in df_train.columns if not col in ['MAKH','DATEIN_DATE', 'MONTHIN', 'TAG_GB']]]
x_test = df_test.loc[:,[col for col in df_test.columns if not col in ['MAKH','DATEIN_DATE', 'MONTHIN', 'TAG_GB']]]
x_val = df_val.loc[:,[col for col in df_val.columns if not col in ['MAKH','DATEIN_DATE', 'MONTHIN', 'TAG_GB']]]

y_train = df_train.loc[:, ['TAG_GB']]
y_test = df_test.loc[:, ['TAG_GB']]
y_val = df_val.loc[:, ['TAG_GB']]

d_train = xgb.DMatrix(x_train, y_train, enable_categorical=True, missing = np.nan)
d_test = xgb.DMatrix(x_test, y_test, enable_categorical=True, missing = np.nan)
d_val = xgb.DMatrix(x_val, y_val, enable_categorical=True, missing = np.nan)

In [39]:
def objective(trial):
    fixed_params = {
        'objective' : 'binary:logistic',
        'random_state' : 42,
        'seed' : 27,
        'nthread': 10
    }
        
    params = {
            'gamma': trial.suggest_float('gamma', 0.5, 8.0),
            'subsample': trial.suggest_float('subsample', 0.5, 0.7),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.7),
            'max_depth': trial.suggest_int('max_depth', 4, 6),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'n_estimators': trial.suggest_int('n_estimators', 300, 900),
            'eta': trial.suggest_float('eta', 0.01, 0.04),
            'lambda': trial.suggest_float('lambda', 0.5, 5),
            'alpha': trial.suggest_float('alpha', 0.0, 2.0),
            'max_delta_step': trial.suggest_int('max_delta_step', 0, 10)
        }

    params_final = {**fixed_params, **params}

    bst = xgb.train(
            {key: value for key, value in params_final.items() if key != 'n_estimators'},
            dtrain = d_train,
            num_boost_round=params_final['n_estimators']
    )

    train_gini = gini_xgb(bst.predict(d_train),d_train)[1]
    val_gini = gini_xgb(bst.predict(d_val),d_val)[1]

    gini_diff = abs(train_gini - val_gini)

    gini_diff_penalty = 1000 if gini_diff > 0.05 else 0

    gini_below_threshold_penalty = 1000 if (train_gini < 0.30 or val_gini < 0.30) else 0

    gini_diff_penalty += gini_diff_penalty
    gini_below_threshold_penalty += gini_below_threshold_penalty

    weight_test_gini = 1.0  
    weight_gini_diff = 1  
    weight_gini_below_threshold = 1

    objective_value = -weight_test_gini * val_gini + weight_gini_diff * gini_diff_penalty + weight_gini_below_threshold * gini_below_threshold_penalty
    return objective_value

study = optuna.create_study(direction='minimize') 
study.optimize(objective, n_trials=100)

print("Best trial:")
print(study.best_trial.params)
print(f"Best objective value: {study.best_value}")

[I 2025-06-19 14:03:11,498] A new study created in memory with name: no-name-0afc4815-c87e-451f-b1d3-c981283a8af2
[I 2025-06-19 14:03:16,663] Trial 0 finished with value: 1999.3224810462905 and parameters: {'gamma': 0.991334652951487, 'subsample': 0.5267518826728101, 'colsample_bytree': 0.533408742366661, 'max_depth': 6, 'min_child_weight': 2, 'n_estimators': 435, 'eta': 0.012642822022624232, 'lambda': 4.600163200061577, 'alpha': 1.0671108233299074, 'max_delta_step': 0}. Best is trial 0 with value: 1999.3224810462905.
[I 2025-06-19 14:03:19,616] Trial 1 finished with value: 1999.3195777100539 and parameters: {'gamma': 2.650215832364538, 'subsample': 0.5704606019515204, 'colsample_bytree': 0.6719874758759171, 'max_depth': 4, 'min_child_weight': 10, 'n_estimators': 351, 'eta': 0.03435973035794833, 'lambda': 2.1202259809051944, 'alpha': 0.384026064353173, 'max_delta_step': 8}. Best is trial 1 with value: 1999.3195777100539.
[I 2025-06-19 14:03:22,807] Trial 2 finished with value: -0.66635

Best trial:
{'gamma': 5.119801593401191, 'subsample': 0.6402492386379844, 'colsample_bytree': 0.5708279509449481, 'max_depth': 4, 'min_child_weight': 7, 'n_estimators': 473, 'eta': 0.030355022688950264, 'lambda': 0.9556007629482202, 'alpha': 0.27093736620109377, 'max_delta_step': 4}
Best objective value: -0.6755341612484469


In [40]:
params_final = {
   'gamma': 5.119801593401191, 'subsample': 0.6402492386379844, 'colsample_bytree': 0.5708279509449481, 'max_depth': 4, 'min_child_weight': 7, 'n_estimators': 473, 'eta': 0.030355022688950264, 'lambda': 0.9556007629482202, 'alpha': 0.27093736620109377, 'max_delta_step': 4,
    'objective' : 'binary:logistic',
    'random_state' : 42,
    'seed' : 27,
    'nthread': 10
}

bst = xgb.train(
            {key: value for key, value in params_final.items() if key != 'n_estimators'},
            dtrain = d_train,
            num_boost_round=params_final['n_estimators']
    )

train_gini = gini_xgb(bst.predict(d_train),d_train)[1]
val_gini = gini_xgb(bst.predict(d_val),d_val)[1]
test_gini = gini_xgb(bst.predict(d_test),d_test)[1]
gini_diff = abs(train_gini - val_gini)

print('Train GINI: {}'.format(train_gini))
print('Val GINI: {}'.format(val_gini))
print('GINI drop: {}'.format(gini_diff))
print('Test GINI: {}'.format(test_gini))

Train GINI: 0.7247412080538185
Val GINI: 0.6755341612484469
GINI drop: 0.04920704680537158
Test GINI: 0.6629815811640907


In [7]:
bst.save_model('recovery_unsecured_dpd361_all_in_one_mob3_28fea.json')

In [41]:
feature_important_weight = bst.get_score(importance_type='weight')
feature_important_gain = bst.get_score(importance_type='gain')
feature_important_cover = bst.get_score(importance_type='cover')

keys = list(feature_important_weight.keys())
weight = list(feature_important_weight.values())
gain = list(feature_important_gain.values())
cover = list(feature_important_cover.values())

fs_df = pd.DataFrame(data={'feature':keys, "weight":weight, 'gain':gain, 'cover':cover})

fs_df.to_csv('feature_importance.csv',index=False)

## Score scaling

In [43]:
train_predict = bst.predict(d_train)
val_predict = bst.predict(d_val)
test_predict = bst.predict(d_test)

In [44]:
train_predict_df = pd.DataFrame(train_predict, columns=['proba'])
val_predict_df = pd.DataFrame(val_predict, columns=['proba'])
test_predict_df = pd.DataFrame(test_predict, columns=['proba'])

In [45]:
df_train = df_train.merge(train_predict_df,left_index=True,right_index=True)
df_val = df_val.merge(val_predict_df,left_index=True,right_index=True)
df_test = df_test.merge(test_predict_df,left_index=True,right_index=True)

In [46]:
pdo = 50
base_score = 500
factor = pdo/np.log(2)
offset = base_score - (factor*np.log(50))

df_train['score'] = df_train['proba'].apply(lambda x: offset + factor*np.log((1-x)/x))
df_val['score'] = df_val['proba'].apply(lambda x: offset + factor*np.log((1-x)/x))
df_test['score'] = df_test['proba'].apply(lambda x: offset + factor*np.log((1-x)/x))

## Model evaluation

### GINI

In [13]:
# Train GINI: 0.7566487786252415
# Val GINI: 0.7080419950061044
# GINI drop: 0.04860678361913706
# Test GINI: 0.6660621393218441

### KS

In [47]:
from functions.KS import calculate_ks_statistic

print('Train KS: {}'.format(calculate_ks_statistic(df_train.score, df_train.TAG_GB)))
print('Val KS: {}'.format(calculate_ks_statistic(df_val.score, df_val.TAG_GB)))
print('Test KS: {}'.format(calculate_ks_statistic(df_test.score, df_test.TAG_GB)))

Train KS: 0.5637534742320187
Val KS: 0.5185387613959043
Test KS: 0.5044412386696242


### PSI

In [48]:
df_train['score_bin']=pd.qcut(df_train.score, 10)

In [49]:
deciles, bin_edges = pd.qcut(df_train.score, 10, retbins=True)

df_val['score_bin'] = pd.cut(df_val.score, bins=bin_edges)
df_test['score_bin'] = pd.cut(df_test.score, bins=bin_edges)

In [50]:
#Calculate bad rate and odd for training set
train_pivot = df_train.pivot_table(index='score_bin', columns='TAG_GB', aggfunc='size', fill_value=0)
train_pivot['BAD_RATE'] = train_pivot[1]/(train_pivot[1]+train_pivot[0])
train_pivot['ODD'] = train_pivot[1]/train_pivot[0]

#Calculate bad rate and odd for validating set
val_pivot = df_val.pivot_table(index='score_bin', columns='TAG_GB', aggfunc='size', fill_value=0)
val_pivot['BAD_RATE'] = val_pivot[1]/(val_pivot[1]+val_pivot[0])
val_pivot['ODD'] = val_pivot[1]/val_pivot[0]

#Calculate bad rate and odd for testing set
test_pivot = df_test.pivot_table(index='score_bin', columns='TAG_GB', aggfunc='size', fill_value=0)
test_pivot['BAD_RATE'] = test_pivot[1]/(test_pivot[1]+test_pivot[0])
test_pivot['ODD'] = test_pivot[1]/test_pivot[0]

train_pivot['DIS']=(train_pivot[0]+train_pivot[1])/(train_pivot[0].sum()+train_pivot[1].sum())
val_pivot['DIS']=(val_pivot[0]+val_pivot[1])/(val_pivot[0].sum()+val_pivot[1].sum())
test_pivot['DIS']=(test_pivot[0]+test_pivot[1])/(test_pivot[0].sum()+test_pivot[1].sum())

#Calculate PSI
train_pivot_sub = train_pivot.loc[:,'DIS']
val_pivot_sub = val_pivot.loc[:,'DIS']
test_pivot_sub = test_pivot.loc[:,'DIS']

train_val = pd.merge(train_pivot_sub,val_pivot_sub,how='inner', left_index=True, right_index=True, suffixes=['_train','_validation'])
train_val['PSI'] = (train_val['DIS_train'] - train_val['DIS_validation'])*np.log(train_val['DIS_train']/train_val['DIS_validation'])

train_test = pd.merge(train_pivot_sub,test_pivot_sub,how='inner', left_index=True, right_index=True, suffixes=['_train','_validation'])
train_test['PSI'] = (train_test['DIS_train'] - train_test['DIS_validation'])*np.log(train_test['DIS_train']/train_test['DIS_validation'])

print('PSI (train vs val): {}'.format(train_val['PSI'].sum()))
print('PSI (train vs test): {}'.format(train_test['PSI'].sum()))

PSI (train vs val): 0.0001954001037524547
PSI (train vs test): 0.0019154400263171783


/tmp/ipykernel_7775/1101814837.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  train_pivot = df_train.pivot_table(index='score_bin', columns='TAG_GB', aggfunc='size', fill_value=0)
/tmp/ipykernel_7775/1101814837.py:7: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  val_pivot = df_val.pivot_table(index='score_bin', columns='TAG_GB', aggfunc='size', fill_value=0)
/tmp/ipykernel_7775/1101814837.py:12: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  test_pivot = df_test.pivot_table(index='score_bin', column

In [51]:
#Calculate bad rate and odd for training set
train_pivot = df_train.pivot_table(index='score_bin', columns='TAG_GB', aggfunc='size', fill_value=0)
train_pivot['BAD_RATE'] = train_pivot[1]/(train_pivot[1]+train_pivot[0])
train_pivot['ODD'] = train_pivot[1]/train_pivot[0]
train_pivot['DIS']=(train_pivot[0]+train_pivot[1])/(train_pivot[0].sum()+train_pivot[1].sum())
train_pivot_sub = train_pivot.loc[:,'DIS']

for i in df_test.MONTHIN.unique():
    df_test_sub = df_test.loc[df_test.MONTHIN==i,:]

    test_pivot = df_test_sub.pivot_table(index='score_bin', columns='TAG_GB', aggfunc='size', fill_value=0)
    test_pivot['BAD_RATE'] = test_pivot[1]/(test_pivot[1]+test_pivot[0])
    test_pivot['ODD'] = test_pivot[1]/test_pivot[0]

    test_pivot['DIS']=(test_pivot[0]+test_pivot[1])/(test_pivot[0].sum()+test_pivot[1].sum())
    test_pivot_sub = test_pivot.loc[:,'DIS']

    train_test = pd.merge(train_pivot_sub,test_pivot_sub,how='inner', left_index=True, right_index=True, suffixes=['_train','_validation'])
    train_test['PSI'] = (train_test['DIS_train'] - train_test['DIS_validation'])*np.log(train_test['DIS_train']/train_test['DIS_validation'])

    print('Test month: {}, PSI: {}'.format(i, train_test['PSI'].sum()))

Test month: 202410, PSI: 0.005420559858155409
Test month: 202501, PSI: 0.0030130683125873304
Test month: 202411, PSI: 0.0017975688257168443
Test month: 202412, PSI: 0.002946925568796827


/tmp/ipykernel_7775/559348544.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  train_pivot = df_train.pivot_table(index='score_bin', columns='TAG_GB', aggfunc='size', fill_value=0)
/tmp/ipykernel_7775/559348544.py:11: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  test_pivot = df_test_sub.pivot_table(index='score_bin', columns='TAG_GB', aggfunc='size', fill_value=0)
/tmp/ipykernel_7775/559348544.py:11: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  test_pivot = df_test_sub.pivot_table(index='score_bin'

In [49]:
df_train

,DATEIN_DATE,MONTHIN,TAG_GB,AVG_LTV_L3M,ca_over_1m_l1m_ct_l3m,ca_min_l12m,sa_min_l12m,sa_median_l12m,casa_min_l12m,sa_iqr_l12m,...,n_dpd_31_90_l3m,cons_delq_0_l3m,cons_delq_90_l3m,cons_delq_90_l12m,cons_delq_decrease_l12m,days_last_ptp_l1m,DIA_CHI,proba,score,score_bin
0,5/20/2024,202405,0,0.528738,NaN,203628.0,0.0,0.0,203628.0,0.0,...,0,0,0,0,0,8.0,1.0,0.022430,490.091849,"(453.07, 495.085]"
1,5/25/2024,202405,0,0.707115,NaN,401423.0,0.0,0.0,401423.0,0.0,...,0,0,0,0,0,NaN,0.0,0.017004,510.470105,"(495.085, 520.179]"
2,5/10/2024,202405,0,0.692592,NaN,217429.0,0.0,0.0,217429.0,0.0,...,0,0,0,0,0,19.0,0.0,0.014189,523.729345,"(520.179, 539.657]"
3,5/25/2024,202405,0,0.543745,NaN,5162.0,0.0,0.0,5162.0,0.0,...,0,0,0,0,0,NaN,0.0,0.007535,569.866377,"(556.129, 570.838]"
4,5/29/2024,202405,0,0.379496,NaN,465004.0,0.0,0.0,465004.0,0.0,...,0,0,0,0,1,0.0,NaN,0.033728,459.828319,"(453.07, 495.085]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111155,4/21/2023,202304,0,0.394103,0.750000,321814.0,0.0,0.0,321814.0,0.0,...,0,0,0,0,0,4.0,3.0,0.013692,526.336430,"(520.179, 539.657]"
111156,4/15/2023,202304,0,0.652208,0.900000,524207.0,0.0,0.0,524207.0,0.0,...,0,0,0,0,0,NaN,1.0,0.006040,585.931482,"(585.598, 601.957]"
111157,4/15/2023,202304,0,0.431215,0.600000,66668.0,0.0,0.0,66668.0,0.0,...,0,0,0,0,0,NaN,0.0,0.007459,570.604731,"(556.129, 570.838]"
111158,4/20/2023,202304,0,0.646248,0.562500,93353.0,0.0,0.0,93353.0,0.0,...,0,0,0,0,0,NaN,3.0,0.014497,522.159662,"(520.179, 539.657]"


In [83]:
cols = ['MAKH','DATEIN_DATE', 'MONTHIN', 'TAG_GB']

df_train_sub=df_train.loc[:,cols]

df_train_sub.to_csv('TUNGTT9_TMP_RECOVERY_PBL_TRAIN.csv',index=False)

In [84]:
cols = ['MAKH','DATEIN_DATE', 'MONTHIN', 'TAG_GB']

df_test_sub=df_test.loc[:,cols]

df_test_sub.to_csv('TUNGTT9_TMP_RECOVERY_PBL_TEST.csv',index=False)

In [85]:
cols = ['MAKH','DATEIN_DATE', 'MONTHIN', 'TAG_GB']
df_val_sub=df_val.loc[:,cols]

df_val_sub.to_csv('TUNGTT9_TMP_RECOVERY_PBL_VAL.csv',index=False)

# Feature analysis

In [87]:
df_train_sub = pd.read_csv('TUNGTT9_TMP_RECOVERY_PBL_TRAIN.csv',sep=',').reset_index(drop=True)

exclude_cols = ['MAKH','DATEIN_DATE', 'MONTHIN', 'TAG_GB']
cols = [col for col in df_train_sub.columns if col not in exclude_cols]

for col in cols:
    df_train_sub[f'{col}_bin'] = pd.qcut(df_train_sub[f'{col}'], 4, duplicates='drop')

In [88]:
df_train['repayment_percentage_l6m_bin'] = pd.qcut(df_train.repayment_percentage_l6m, 4, duplicates='drop')

repayment_percentage_l6m_pivot = df_train.pivot_table(index='repayment_percentage_l6m_bin', columns='TAG_GB', aggfunc='size', fill_value=0)
repayment_percentage_l6m_pivot['BAD_RATE'] = repayment_percentage_l6m_pivot[1]/(repayment_percentage_l6m_pivot[1]+repayment_percentage_l6m_pivot[0])
repayment_percentage_l6m_pivot=repayment_percentage_l6m_pivot.reset_index(drop=False)
# repayment_percentage_l6m_pivot
repayment_percentage_l6m_pivot.to_csv('repayment_percentage_l6m_pivot.csv', index=False)

AttributeError: 'DataFrame' object has no attribute 'repayment_percentage_l6m'

In [26]:
import pandas as pd

# Calculate the bad rate pivot table
pivot_table = pd.pivot_table(
    df_train_sub,
    index='age_bin',
    columns='repayment_percentage_l6m_bin',
    values='tag_good_bad',
    aggfunc=lambda x: (x == 1).sum() / len(x)
)

# Optional: format for clarity
pivot_table = pivot_table.round(4)

# pivot_table
pivot_table.to_csv('age_ct_repayment_percentage_l6m.csv')

/tmp/ipykernel_3371/4245632668.py:4: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_table = pd.pivot_table(


In [24]:
# Calculate the bad rate pivot table
pivot_table = pd.pivot_table(
    df_train_sub,
    index='trans_in_amt_cv_l3m_bin',
    columns='repayment_percentage_l6m_bin',
    values='tag_good_bad',
    aggfunc=lambda x: (x == 1).sum() / len(x)
)

# Optional: format for clarity
pivot_table = pivot_table.round(4)

# pivot_table
pivot_table.to_csv('trans_in_amt_cv_l3m_ct_repayment_percentage_l6m.csv')

/tmp/ipykernel_20175/2825071196.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_table = pd.pivot_table(


In [27]:
# Calculate the bad rate pivot table
pivot_table = pd.pivot_table(
    df_train_sub,
    index='repayment_percentage_l1m_bin',
    columns='m_since_max_payment_l6m_bin',
    values='tag_good_bad',
    aggfunc=lambda x: (x == 1).sum() / len(x)
)

# Optional: format for clarity
pivot_table = pivot_table.round(4)

pivot_table
# pivot_table.to_csv('repayment_percentage_l1m_ct_m_since_max_payment_l6m.csv')

/tmp/ipykernel_3371/2728673713.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_table = pd.pivot_table(


m_since_max_payment_l6m_bin,"(-0.001, 1.0]","(1.0, 3.0]","(3.0, 4.0]","(4.0, 5.0]"
repayment_percentage_l1m_bin,,,,
"(-0.001, 1.0]",0.0595,0.0427,0.0381,0.0362


In [27]:
# Calculate the bad rate pivot table
pivot_table = pd.pivot_table(
    df_train_sub,
    index='repayment_percentage_l3m_bin',
    columns='count_activity_cv_l3m_bin',
    values='tag_good_bad',
    aggfunc=lambda x: (x == 1).sum() / len(x)
)

# Optional: format for clarity
pivot_table = pivot_table.round(4)

# pivot_table
pivot_table.to_csv('repayment_percentage_l3m_ct_count_activity_cv_l3m.csv')

/tmp/ipykernel_20175/4237662857.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_table = pd.pivot_table(


In [29]:
# Calculate the bad rate pivot table
pivot_table = pd.pivot_table(
    df_train_sub,
    index='total_ost_bin',
    columns='repayment_percentage_l3m_bin',
    values='tag_good_bad',
    aggfunc=lambda x: (x == 1).sum() / len(x)
)

# Optional: format for clarity
pivot_table = pivot_table.round(4)

pivot_table
# pivot_table.to_csv('total_ost_ct_repayment_percentage_l3m.csv')

/tmp/ipykernel_3371/3708890818.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_table = pd.pivot_table(


repayment_percentage_l3m_bin,"(-0.001, 1.0]"
total_ost_bin,
"(-0.00099979, 14.0]",0.0492
"(14.0, 44.0]",0.0280
"(44.0, 82.104]",0.0151
"(82.104, 11989.171]",0.0079


In [33]:
# Calculate the bad rate pivot table
pivot_table = pd.pivot_table(
    df_train_sub,
    index='trans_out_amt_iqr_l6m_bin',
    columns='total_ost_bin',
    values='tag_good_bad',
    aggfunc=lambda x: (x == 1).sum() / len(x)
)

# Optional: format for clarity
pivot_table = pivot_table.round(4)

# pivot_table
pivot_table.to_csv('trans_out_amt_iqr_l6m_ct_total_ost.csv')

/tmp/ipykernel_3371/219964078.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_table = pd.pivot_table(


In [31]:
# Calculate the bad rate pivot table
pivot_table = pd.pivot_table(
    df_train_sub,
    index='total_ost_bin',
    columns='age_bin',
    values='tag_good_bad',
    aggfunc=lambda x: (x == 1).sum() / len(x)
)

# Optional: format for clarity
pivot_table = pivot_table.round(4)

# pivot_table
pivot_table.to_csv('total_ost_ct_age.csv')

/tmp/ipykernel_3371/1507466639.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_table = pd.pivot_table(


In [35]:
# Calculate the bad rate pivot table
pivot_table = pd.pivot_table(
    df_train_sub,
    index='m_since_last_payment_l6m_bin',
    columns='m_since_max_payment_l6m_bin',
    values='tag_good_bad',
    aggfunc=lambda x: (x == 1).sum() / len(x)
)

# Optional: format for clarity
pivot_table = pivot_table.round(4)

# pivot_table
pivot_table.to_csv('m_since_last_payment_l6m_ct_m_since_max_payment_l6m.csv')

/tmp/ipykernel_3371/2866174604.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_table = pd.pivot_table(


In [36]:
# Calculate the bad rate pivot table
pivot_table = pd.pivot_table(
    df_train_sub,
    index='trans_in_amt_min_l6m_bin',
    columns='total_ost_bin',
    values='tag_good_bad',
    aggfunc=lambda x: (x == 1).sum() / len(x)
)

# Optional: format for clarity
pivot_table = pivot_table.round(4)

pivot_table
# pivot_table.to_csv('trans_in_amt_min_l6m_ct_total_ost.csv')

/tmp/ipykernel_3371/3382179354.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_table = pd.pivot_table(


total_ost_bin,"(-0.00099979, 14.0]","(14.0, 44.0]","(44.0, 82.104]","(82.104, 11989.171]"
trans_in_amt_min_l6m_bin,,,,
"(-0.001, 20333.209]",0.0782,0.0463,0.0234,0.0173


In [38]:
# Calculate the bad rate pivot table
pivot_table = pd.pivot_table(
    df_train_sub,
    index='repayment_percentage_l6m_bin',
    columns='count_session_id_avg_l3m_bin',
    values='tag_good_bad',
    aggfunc=lambda x: (x == 1).sum() / len(x)
)

# Optional: format for clarity
pivot_table = pivot_table.round(4)

# pivot_table
pivot_table.to_csv('repayment_percentage_l6m_ct_count_session_id_avg_l3m.csv')

/tmp/ipykernel_20175/1094189453.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_table = pd.pivot_table(


In [38]:
# Calculate the bad rate pivot table
pivot_table = pd.pivot_table(
    df_train_sub,
    index='trans_in_num_median_l3m_bin',
    columns='age_bin',
    values='tag_good_bad',
    aggfunc=lambda x: (x == 1).sum() / len(x)
)

# Optional: format for clarity
pivot_table = pivot_table.round(4)

# pivot_table
pivot_table.to_csv('trans_in_num_median_l3m_ct_age.csv')

/tmp/ipykernel_3371/2972268290.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_table = pd.pivot_table(


In [40]:
# Calculate the bad rate pivot table
pivot_table = pd.pivot_table(
    df_train_sub,
    index='trans_out_amt_std_l3m_bin',
    columns='total_ost_bin',
    values='tag_good_bad',
    aggfunc=lambda x: (x == 1).sum() / len(x)
)

# Optional: format for clarity
pivot_table = pivot_table.round(4)

# pivot_table
pivot_table.to_csv('trans_out_amt_std_l3m_ct_total_ost.csv')

/tmp/ipykernel_3371/3221889833.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_table = pd.pivot_table(


In [42]:
# Calculate the bad rate pivot table
pivot_table = pd.pivot_table(
    df_train_sub,
    index='trans_out_amt_q75_l3m_bin',
    columns='dpd_bin',
    values='tag_good_bad',
    aggfunc=lambda x: (x == 1).sum() / len(x)
)

# Optional: format for clarity
pivot_table = pivot_table.round(4)

# pivot_table
pivot_table.to_csv('trans_out_amt_q75_l3m_ct_dpd.csv')

/tmp/ipykernel_3371/2115959963.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_table = pd.pivot_table(


In [43]:
# Calculate the bad rate pivot table
pivot_table = pd.pivot_table(
    df_train_sub,
    index='repayment_percentage_l3m_bin',
    columns='m_since_last_payment_l6m_bin',
    values='tag_good_bad',
    aggfunc=lambda x: (x == 1).sum() / len(x)
)

# Optional: format for clarity
pivot_table = pivot_table.round(4)

pivot_table
# pivot_table.to_csv('repayment_percentage_l3m_ct_m_since_last_payment_l6m.csv')

/tmp/ipykernel_3371/2275656235.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_table = pd.pivot_table(


m_since_last_payment_l6m_bin,"(-0.001, 2.0]","(2.0, 4.0]","(4.0, 5.0]"
repayment_percentage_l3m_bin,,,
"(-0.001, 1.0]",0.0615,0.0216,0.0184


In [45]:
# Calculate the bad rate pivot table
pivot_table = pd.pivot_table(
    df_train_sub,
    index='total_ost_bin',
    columns='m_since_max_payment_l6m_bin',
    values='tag_good_bad',
    aggfunc=lambda x: (x == 1).sum() / len(x)
)

# Optional: format for clarity
pivot_table = pivot_table.round(4)

# pivot_table
pivot_table.to_csv('total_ost_ct_m_since_max_payment_l6m.csv')

/tmp/ipykernel_3371/4250675471.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_table = pd.pivot_table(


In [47]:
# Calculate the bad rate pivot table
pivot_table = pd.pivot_table(
    df_train_sub,
    index='trans_in_num_iqr_l6m_bin',
    columns='repayment_percentage_l6m_bin',
    values='tag_good_bad',
    aggfunc=lambda x: (x == 1).sum() / len(x)
)

# Optional: format for clarity
pivot_table = pivot_table.round(4)

# pivot_table
pivot_table.to_csv('trans_in_num_iqr_l6m_ct_repayment_percentage_l6m.csv')

/tmp/ipykernel_3371/3512435169.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_table = pd.pivot_table(


In [50]:
# Calculate the bad rate pivot table
pivot_table = pd.pivot_table(
    df_train_sub,
    index='trans_out_num_max_l6m_bin',
    columns='repayment_percentage_l6m_bin',
    values='tag_good_bad',
    aggfunc=lambda x: (x == 1).sum() / len(x)
)

# Optional: format for clarity
pivot_table = pivot_table.round(4)

# pivot_table
pivot_table.to_csv('trans_out_num_max_l6m_ct_repayment_percentage_l6m.csv')

/tmp/ipykernel_3371/2563301920.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_table = pd.pivot_table(


In [52]:
# Calculate the bad rate pivot table
pivot_table = pd.pivot_table(
    df_train_sub,
    index='trans_in_amt_max_l6m_bin',
    columns='repayment_percentage_l6m_bin',
    values='tag_good_bad',
    aggfunc=lambda x: (x == 1).sum() / len(x)
)

# Optional: format for clarity
pivot_table = pivot_table.round(4)

# pivot_table
pivot_table.to_csv('trans_in_amt_max_l6m_ct_repayment_percentage_l6m.csv')

/tmp/ipykernel_3371/1949542129.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_table = pd.pivot_table(


In [54]:
# Calculate the bad rate pivot table
pivot_table = pd.pivot_table(
    df_train_sub,
    index='trans_in_num_iqr_l6m_bin',
    columns='total_ost_bin',
    values='tag_good_bad',
    aggfunc=lambda x: (x == 1).sum() / len(x)
)

# Optional: format for clarity
pivot_table = pivot_table.round(4)

# pivot_table
pivot_table.to_csv('trans_in_num_iqr_l6m_ct_total_ost.csv')

/tmp/ipykernel_3371/1198992032.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_table = pd.pivot_table(


In [55]:
# Calculate the bad rate pivot table
pivot_table = pd.pivot_table(
    df_train_sub,
    index='trans_out_amt_min_l6m_bin',
    columns='dpd_bin',
    values='tag_good_bad',
    aggfunc=lambda x: (x == 1).sum() / len(x)
)

# Optional: format for clarity
pivot_table = pivot_table.round(4)

pivot_table
# pivot_table.to_csv('trans_out_amt_min_l6m_ct_dpd.csv')

/tmp/ipykernel_3371/1492062497.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_table = pd.pivot_table(


dpd_bin,"(360.999, 495.0]","(495.0, 648.0]","(648.0, 954.0]","(954.0, 6355.0]"
trans_out_amt_min_l6m_bin,,,,
"(-0.001, 20325.764]",0.042,0.0434,0.0466,0.0348


In [56]:
# Calculate the bad rate pivot table
pivot_table = pd.pivot_table(
    df_train_sub,
    index='trans_in_num_min_l6m_bin',
    columns='age_bin',
    values='tag_good_bad',
    aggfunc=lambda x: (x == 1).sum() / len(x)
)

# Optional: format for clarity
pivot_table = pivot_table.round(4)

pivot_table
# pivot_table.to_csv('trans_in_num_min_l6m_ct_age.csv')

/tmp/ipykernel_3371/372727902.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_table = pd.pivot_table(


age_bin,"(3.999, 28.0]","(28.0, 33.0]","(33.0, 39.0]","(39.0, 124.0]"
trans_in_num_min_l6m_bin,,,,
"(-0.001, 589.0]",0.044,0.0378,0.0389,0.0497


In [58]:
# Calculate the bad rate pivot table
pivot_table = pd.pivot_table(
    df_train_sub,
    index='trans_in_amt_iqr_l6m_bin',
    columns='repayment_percentage_l6m_bin',
    values='tag_good_bad',
    aggfunc=lambda x: (x == 1).sum() / len(x)
)

# Optional: format for clarity
pivot_table = pivot_table.round(4)

# pivot_table
pivot_table.to_csv('trans_in_amt_iqr_l6m_ct_repayment_percentage_l6m.csv')

/tmp/ipykernel_3371/2155377415.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_table = pd.pivot_table(


In [60]:
# Calculate the bad rate pivot table
pivot_table = pd.pivot_table(
    df_train_sub,
    index='dpd_bin',
    columns='age_bin',
    values='tag_good_bad',
    aggfunc=lambda x: (x == 1).sum() / len(x)
)

# Optional: format for clarity
pivot_table = pivot_table.round(4)

# pivot_table
pivot_table.to_csv('dpd_ct_age.csv')

/tmp/ipykernel_3371/3049566893.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_table = pd.pivot_table(


In [62]:
# Calculate the bad rate pivot table
pivot_table = pd.pivot_table(
    df_train_sub,
    index='trans_out_amt_std_l3m_bin',
    columns='dpd_bin',
    values='tag_good_bad',
    aggfunc=lambda x: (x == 1).sum() / len(x)
)

# Optional: format for clarity
pivot_table = pivot_table.round(4)

# pivot_table
pivot_table.to_csv('trans_out_amt_std_l3m_ct_dpd.csv')

/tmp/ipykernel_3371/635115357.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_table = pd.pivot_table(


In [65]:
df_train['age_bin'] = pd.qcut(df_train.age, 4, duplicates='drop')

age_pivot = df_train.pivot_table(index='age_bin', columns='tag_good_bad', aggfunc='size', fill_value=0)
age_pivot['BAD_RATE'] = age_pivot[1]/(age_pivot[1]+age_pivot[0])
age_pivot=age_pivot.reset_index(drop=False)
# age_pivot
age_pivot.to_csv('age_pivot.csv', index=False)

/tmp/ipykernel_3371/3057596293.py:3: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  age_pivot = df_train.pivot_table(index='age_bin', columns='tag_good_bad', aggfunc='size', fill_value=0)


In [67]:
df_train['trans_out_amt_iqr_l6m_bin'] = pd.qcut(df_train.trans_out_amt_iqr_l6m, 4, duplicates='drop')

trans_out_amt_iqr_l6m_pivot = df_train.pivot_table(index='trans_out_amt_iqr_l6m_bin', columns='tag_good_bad', aggfunc='size', fill_value=0)
trans_out_amt_iqr_l6m_pivot['BAD_RATE'] = trans_out_amt_iqr_l6m_pivot[1]/(trans_out_amt_iqr_l6m_pivot[1]+trans_out_amt_iqr_l6m_pivot[0])
trans_out_amt_iqr_l6m_pivot=trans_out_amt_iqr_l6m_pivot.reset_index(drop=False)
# trans_out_amt_iqr_l6m_pivot
trans_out_amt_iqr_l6m_pivot.to_csv('trans_out_amt_iqr_l6m_pivot.csv', index=False)

/tmp/ipykernel_3371/2359554724.py:3: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  trans_out_amt_iqr_l6m_pivot = df_train.pivot_table(index='trans_out_amt_iqr_l6m_bin', columns='tag_good_bad', aggfunc='size', fill_value=0)


In [68]:
# Calculate the bad rate pivot table
pivot_table = pd.pivot_table(
    df_train_sub,
    index='trans_in_num_iqr_l3m_bin',
    columns='age_bin',
    values='tag_good_bad',
    aggfunc=lambda x: (x == 1).sum() / len(x)
)

# Optional: format for clarity
pivot_table = pivot_table.round(4)

# pivot_table
pivot_table.to_csv('trans_in_num_iqr_l3m_ct_age.csv')

/tmp/ipykernel_3371/233617570.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_table = pd.pivot_table(


In [70]:
# Calculate the bad rate pivot table
pivot_table = pd.pivot_table(
    df_train_sub,
    index='trans_out_num_iqr_l6m_bin',
    columns='dpd_bin',
    values='tag_good_bad',
    aggfunc=lambda x: (x == 1).sum() / len(x)
)

# Optional: format for clarity
pivot_table = pivot_table.round(4)

# pivot_table
pivot_table.to_csv('trans_out_num_iqr_l6m_ct_dpd.csv')

/tmp/ipykernel_3371/1374843707.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_table = pd.pivot_table(
